# Ejercicio 1: Clasificador Bayesiano Óptimo (Datos Discretos)

## Enunciado
Para los siguientes datos, construir el clasificador bayesiano óptimo. Indicar la regla de clasificación y hallar el error del clasificador.

| | X=1 | X=2 | X=3 | X=4 |
|---|---|---|---|---|
| Y=0 | 0.09 | 0.16 | 0.17 | 0.05 |
| Y=1 | 0.15 | 0.07 | 0.10 | 0.21 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Definición de los datos

Los datos representan probabilidades conjuntas $P(X=x, Y=y)$.

In [ ]:
# Probabilidades conjuntas P(X, Y)
# Filas: Y (0, 1)
# Columnas: X (1, 2, 3, 4)

P_XY = np.array([
    [0.09, 0.16, 0.17, 0.05],  # Y = 0
    [0.15, 0.07, 0.10, 0.21]   # Y = 1
])

# Valores de X
X_values = [1, 2, 3, 4]

# Crear DataFrame para visualización
df = pd.DataFrame(P_XY, index=['Y=0', 'Y=1'], columns=['X=1', 'X=2', 'X=3', 'X=4'])
print("Tabla de probabilidades conjuntas P(X, Y):")
df

## 2. Cálculo de probabilidades marginales

### Probabilidades a priori $P(Y)$
$$P(Y=y) = \sum_{x} P(X=x, Y=y)$$

In [ ]:
# Probabilidades marginales de Y (a priori)
P_Y0 = np.sum(P_XY[0, :])  # P(Y=0)
P_Y1 = np.sum(P_XY[1, :])  # P(Y=1)

print(f"P(Y=0) = {P_Y0:.2f}")
print(f"P(Y=1) = {P_Y1:.2f}")
print(f"Suma total = {P_Y0 + P_Y1:.2f} (debe ser 1)")

### Probabilidades marginales $P(X)$
$$P(X=x) = \sum_{y} P(X=x, Y=y)$$

In [ ]:
# Probabilidades marginales de X
P_X = np.sum(P_XY, axis=0)  # Suma sobre Y

print("Probabilidades marginales P(X):")
for i, x in enumerate(X_values):
    print(f"P(X={x}) = {P_X[i]:.2f}")

## 3. Cálculo de probabilidades posteriores $P(Y|X)$

Usando el Teorema de Bayes:
$$P(Y=y|X=x) = \frac{P(X=x, Y=y)}{P(X=x)}$$

In [ ]:
# Probabilidades posteriores P(Y|X)
P_Y0_given_X = P_XY[0, :] / P_X  # P(Y=0|X)
P_Y1_given_X = P_XY[1, :] / P_X  # P(Y=1|X)

print("Probabilidades posteriores:")
print("="*50)
for i, x in enumerate(X_values):
    print(f"X={x}:")
    print(f"  P(Y=0|X={x}) = {P_XY[0,i]:.2f} / {P_X[i]:.2f} = {P_Y0_given_X[i]:.4f}")
    print(f"  P(Y=1|X={x}) = {P_XY[1,i]:.2f} / {P_X[i]:.2f} = {P_Y1_given_X[i]:.4f}")
    print()

In [ ]:
# Tabla resumen de probabilidades posteriores
df_posterior = pd.DataFrame({
    'X': X_values,
    'P(Y=0|X)': P_Y0_given_X,
    'P(Y=1|X)': P_Y1_given_X
})
df_posterior.set_index('X', inplace=True)
print("Tabla de probabilidades posteriores:")
df_posterior

## 4. Regla de clasificación bayesiana óptima

El clasificador bayesiano óptimo asigna la clase con mayor probabilidad posterior:

$$\hat{Y}(x) = \arg\max_{y \in \{0,1\}} P(Y=y|X=x)$$

Equivalentemente, clasificamos como $Y=1$ si $P(Y=1|X=x) > P(Y=0|X=x)$, es decir:
$$\hat{Y}(x) = \begin{cases} 1 & \text{si } P(Y=1|X=x) > 0.5 \\ 0 & \text{en otro caso} \end{cases}$$

In [ ]:
# Regla de clasificación
clasificacion = []

print("REGLA DE CLASIFICACIÓN BAYESIANA ÓPTIMA")
print("="*60)
for i, x in enumerate(X_values):
    if P_Y1_given_X[i] > P_Y0_given_X[i]:
        clase = 1
        decision = f"P(Y=1|X={x}) = {P_Y1_given_X[i]:.4f} > P(Y=0|X={x}) = {P_Y0_given_X[i]:.4f}"
    else:
        clase = 0
        decision = f"P(Y=0|X={x}) = {P_Y0_given_X[i]:.4f} > P(Y=1|X={x}) = {P_Y1_given_X[i]:.4f}"
    
    clasificacion.append(clase)
    print(f"Si X={x}: {decision}")
    print(f"         → Clasificar como Y={clase}")
    print()

In [ ]:
# Resumen de la regla de clasificación
print("\n" + "="*60)
print("RESUMEN DE LA REGLA DE CLASIFICACIÓN:")
print("="*60)
for i, x in enumerate(X_values):
    print(f"h(X={x}) = {clasificacion[i]}")

# En forma compacta
print("\nEn forma compacta:")
print("h(x) = 0  si x ∈ {2, 3}")
print("h(x) = 1  si x ∈ {1, 4}")

## 5. Cálculo del error del clasificador

El error del clasificador bayesiano óptimo (error de Bayes) se calcula como:

$$\text{Error} = \sum_{x} P(X=x) \cdot \min\{P(Y=0|X=x), P(Y=1|X=x)\}$$

O equivalentemente:
$$\text{Error} = \sum_{x} \min\{P(X=x, Y=0), P(X=x, Y=1)\}$$

In [ ]:
# Cálculo del error de Bayes
error_por_x = []

print("Cálculo del error por cada valor de X:")
print("="*60)

for i, x in enumerate(X_values):
    # Error = P(clasificar incorrectamente | X=x) * P(X=x)
    # = min{P(Y=0,X=x), P(Y=1,X=x)}
    error_x = min(P_XY[0, i], P_XY[1, i])
    error_por_x.append(error_x)
    
    print(f"X={x}: min{{P(X={x},Y=0), P(X={x},Y=1)}} = min{{{P_XY[0,i]:.2f}, {P_XY[1,i]:.2f}}} = {error_x:.2f}")

error_total = sum(error_por_x)
print(f"\nError total = {' + '.join([f'{e:.2f}' for e in error_por_x])} = {error_total:.2f}")

In [ ]:
# Verificación usando la otra fórmula
print("\nVerificación usando P(X) * min{P(Y|X)}:")
print("="*60)

error_verificacion = 0
for i, x in enumerate(X_values):
    min_prob = min(P_Y0_given_X[i], P_Y1_given_X[i])
    error_x = P_X[i] * min_prob
    error_verificacion += error_x
    print(f"X={x}: P(X={x}) × min{{P(Y|X={x})}} = {P_X[i]:.2f} × {min_prob:.4f} = {error_x:.4f}")

print(f"\nError total (verificación) = {error_verificacion:.4f}")

## 6. Visualización

In [ ]:
# Visualización de las probabilidades posteriores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Probabilidades posteriores
x_pos = np.arange(len(X_values))
width = 0.35

bars1 = axes[0].bar(x_pos - width/2, P_Y0_given_X, width, label='P(Y=0|X)', color='steelblue', alpha=0.8)
bars2 = axes[0].bar(x_pos + width/2, P_Y1_given_X, width, label='P(Y=1|X)', color='coral', alpha=0.8)

axes[0].axhline(y=0.5, color='gray', linestyle='--', linewidth=1, label='Umbral = 0.5')
axes[0].set_xlabel('X', fontsize=12)
axes[0].set_ylabel('Probabilidad posterior', fontsize=12)
axes[0].set_title('Probabilidades Posteriores P(Y|X)', fontsize=14)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f'X={x}' for x in X_values])
axes[0].legend()
axes[0].set_ylim(0, 1)

# Añadir valores sobre las barras
for bar in bars1:
    height = bar.get_height()
    axes[0].annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                     xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    axes[0].annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                     xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

# Gráfico 2: Clasificación y errores
colors = ['coral' if c == 1 else 'steelblue' for c in clasificacion]
axes[1].bar(x_pos, error_por_x, color=colors, alpha=0.8, edgecolor='black')
axes[1].set_xlabel('X', fontsize=12)
axes[1].set_ylabel('Error de clasificación', fontsize=12)
axes[1].set_title(f'Error por valor de X\n(Error total = {error_total:.2f})', fontsize=14)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'X={x}\n(→Y={c})' for x, c in zip(X_values, clasificacion)])

# Añadir valores sobre las barras
for i, e in enumerate(error_por_x):
    axes[1].annotate(f'{e:.2f}', xy=(i, e), xytext=(0, 3), 
                     textcoords='offset points', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig('ejercicio1_clasificador_bayesiano.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Conclusiones

### Regla de Clasificación Bayesiana Óptima:

$$h(x) = \begin{cases} 
0 & \text{si } x \in \{2, 3\} \\
1 & \text{si } x \in \{1, 4\}
\end{cases}$$

### Error del Clasificador:

$$\text{Error de Bayes} = 0.09 + 0.07 + 0.10 + 0.05 = 0.31$$

El clasificador bayesiano óptimo tiene un **error del 31%** (o equivalentemente, una **precisión del 69%**).

Este es el menor error posible para cualquier clasificador dado este conjunto de datos, ya que el clasificador de Bayes minimiza la probabilidad de error.

In [ ]:
# Resumen final
print("="*60)
print("RESUMEN DEL CLASIFICADOR BAYESIANO ÓPTIMO")
print("="*60)
print("\nRegla de clasificación:")
print("   h(x) = 0  si x ∈ {2, 3}")
print("   h(x) = 1  si x ∈ {1, 4}")
print(f"\nError de Bayes: {error_total:.2f} ({error_total*100:.0f}%)")
print(f"Precisión: {1-error_total:.2f} ({(1-error_total)*100:.0f}%)")
print("="*60)